# Chapitre 15 - Exercices 15.1 à 15.4

- **15.1** : variance d'une moyenne de variables équicorrélées, et le cas $\rho<0$ ;
- **15.2** : erreur *out-of-bag* et validation croisée « leave-one-out » ;
- **15.3** : le modèle de simulation de Mease et Wyner (figure 15.7) et son erreur de Bayes ;
- **15.4** : corrélation entre deux moyennes bootstrap, et inefficacité du bagging pour
  une statistique linéaire.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch15_forets as rf
set_style()

## Exercice 15.1 : Variance d'une moyenne de variables corrélées

**Énoncé.** Établir la formule (15.1) : si $T_1,\dots,T_B$ ont toutes la variance
$\sigma^2$ et une corrélation deux à deux égale à $\rho$, alors
$$\operatorname{Var}\Big(\frac1B\sum_{b=1}^BT_b\Big)=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2.$$
Cette formule semble mise en défaut lorsque $\rho<0$ ; expliquer le problème.

### Rappels

- Covariance : $\operatorname{Cov}(U,V)=\mathbb E[(U-\mathbb EU)(V-\mathbb EV)]$ ; elle est
  **bilinéaire** et symétrique, et $\operatorname{Cov}(U,U)=\operatorname{Var}U$.
- Corrélation : $\operatorname{corr}(U,V)=\operatorname{Cov}(U,V)/\sqrt{\operatorname{Var}U\operatorname{Var}V}$,
  donc ici $\operatorname{Cov}(T_b,T_{b'})=\rho\sigma^2$ pour $b\neq b'$.
- Pour un vecteur aléatoire $T=(T_1,\dots,T_B)^T$ de matrice de covariance $\Sigma$ et un
  vecteur fixe $a$, $\operatorname{Var}(a^TT)=a^T\Sigma a\ge0$ : une matrice de covariance
  est toujours **symétrique positive** (semi-définie positive).

### Démonstration de (15.1)

Par bilinéarité de la covariance,
$$\operatorname{Var}\Big(\sum_bT_b\Big)=\operatorname{Cov}\Big(\sum_bT_b,\sum_{b'}T_{b'}\Big)=\sum_{b}\sum_{b'}\operatorname{Cov}(T_b,T_{b'}).$$
La double somme comporte $B$ termes diagonaux ($b=b'$), chacun égal à $\sigma^2$, et
$B(B-1)$ termes hors diagonale, chacun égal à $\rho\sigma^2$. Donc
$$\operatorname{Var}\Big(\sum_bT_b\Big)=B\sigma^2+B(B-1)\rho\sigma^2 .$$
Comme $\operatorname{Var}(cU)=c^2\operatorname{Var}U$, on divise par $B^2$ :
$$\operatorname{Var}\Big(\frac1B\sum_bT_b\Big)=\frac{\sigma^2}{B}+\frac{B-1}{B}\rho\sigma^2
=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2=\frac{\sigma^2}{B}\big(1+(B-1)\rho\big).\qquad\blacksquare$$

### Le cas $\rho<0$

Si $\rho<0$ est fixé, le terme $\rho\sigma^2$ est négatif et la formule donne une
variance **négative** dès que $1+(B-1)\rho<0$, c'est-à-dire dès que $B>1-1/\rho$. Une
variance ne pouvant être négative, quelque chose ne va pas. Le diagnostic : **l'hypothèse
elle-même est impossible**. On ne peut pas avoir $B$ variables deux à deux corrélées
négativement avec la même corrélation $\rho$ quand $B$ est grand.

Pour le voir, écrivons la matrice de covariance
$\Sigma=\sigma^2\big[(1-\rho)\mathbf I+\rho\,\mathbf 1\mathbf 1^T\big]$, où $\mathbf 1$ est
le vecteur de $B$ uns. Cherchons ses valeurs propres :

- $\Sigma\mathbf 1=\sigma^2\big[(1-\rho)\mathbf 1+\rho B\mathbf 1\big]=\sigma^2(1+(B-1)\rho)\,\mathbf 1$
  (car $\mathbf 1^T\mathbf 1=B$) : valeur propre $\lambda_1=\sigma^2(1+(B-1)\rho)$ ;
- pour tout $v\perp\mathbf 1$ (sous-espace de dimension $B-1$), $\mathbf 1^Tv=0$ donc
  $\Sigma v=\sigma^2(1-\rho)v$ : valeur propre $\sigma^2(1-\rho)$, de multiplicité $B-1$.

$\Sigma$ est positive si et seulement si ses valeurs propres sont $\ge0$, soit
$$-\frac1{B-1}\le\rho\le1 .$$